# Exercițiu Laborator 3 — Rețele Bayesiene

## Enunțul problemei
Un sistem de clasificare a e-mail-urilor folosește o rețea bayesiană cu variabilele binare:
* $S \in \{0, 1\}$: e-mail-ul este spam ($S=1$) sau non-spam ($S=0$);
* $O \in \{0, 1\}$: e-mail-ul conține cuvântul „ofertă” ($O=1$) sau nu ($O=0$);
* $L \in \{0, 1\}$: e-mail-ul conține link-uri ($L=1$) sau nu ($L=0$);
* $M \in \{0, 1\}$: e-mail-ul are o lungime mare ($M=1$) sau nu ($M=0$).

**Structura rețelei (DAG):**
* $S \to O$
* $S \to L$
* $S \to M$ și $L \to M$

**Tabelele de probabilități (CPD):**
* $P(S=1) = 0.4, \quad P(S=0) = 0.6$
* $P(O=1 \mid S=1) = 0.7, \quad P(O=1 \mid S=0) = 0.1$
* $P(L=1 \mid S=1) = 0.8, \quad P(L=1 \mid S=0) = 0.3$
* $P(M=1 \mid S=1, L=1) = 0.9, \quad P(M=1 \mid S=1, L=0) = 0.5$
* $P(M=1 \mid S=0, L=1) = 0.6, \quad P(M=1 \mid S=0, L=0) = 0.2$

**Cerințe:**
* **a)** Identificați independențele din rețea.
* **b)** Determinați cum clasifică rețeaua bayesiană e-mail-urile în funcție de atributele $O$, $L$ și $M$.

## a) Identificarea independențelor din rețea (d-separare & proprietatea locală Markov)

Graful orientat aciclic (DAG) are mulțimea arcelor $E = \{(S, O), (S, L), (S, M), (L, M)\}$.

1. **Independențe necondiționate (marginale):**
   * **Nu există nicio pereche de variabile marginal independente** ($X \not\perp\!\!\!\perp Y$ pentru orice $X \neq Y$). Nodul $S$ este cauză comună pentru $O, L, M$, iar $L$ influențează direct pe $M$, astfel încât între orice două noduri există cel puțin un drum activ (neblocat de un *collider* neobservat).

2. **Independențe condiționate:**
   Aplicând **proprietatea locală Markov** (fiecare nod este condiționat independent de non-descendenții săi, dați fiind părinții săi):
   * Pentru nodul $O$: $\text{Parents}(O) = \{S\}$, iar $\text{NonDescendants}(O) = \{L, M\}$. Prin urmare:
     $$O \perp\!\!\!\perp \{L, M\} \mid S.$$
   * De aici rezultă imediat următoarele independențe condiționate:
     * **$O \perp\!\!\!\perp L \mid S$** (structură de tip *fork* / cauză comună $O \leftarrow S \to L$: condiționarea pe $S$ blochează calea dintre $O$ și $L$);
     * **$O \perp\!\!\!\perp M \mid S$** (ambele căi dintre $O$ și $M$, anume $O \leftarrow S \to M$ și $O \leftarrow S \to L \to M$, pleacă din $S$ în mod *chain/fork*, deci sunt blocate când $S$ este observat);
     * **$O \perp\!\!\!\perp M \mid \{S, L\}$** (observarea suplimentară a lui $L$ lasă în continuare nodul $S$ să blocheze ambele căi dintre $O$ și $M$).

3. **Dependențe care se păstrează:**
   * $S$ și $L$ sunt conectate direct ($S \to L$), deci $S \not\perp\!\!\!\perp L$ și $S \not\perp\!\!\!\perp L \mid M$.
   * $L$ și $M$ sunt conectate direct ($L \to M$), deci $L \not\perp\!\!\!\perp M$ și $L \not\perp\!\!\!\perp M \mid S$.

## b) Clasificarea e-mail-urilor în funcție de atributele $(O, L, M)$

Distribuția comună a rețelei Bayesiene se factorizează astfel:
$$
P(S, O, L, M) = P(S) \cdot P(O \mid S) \cdot P(L \mid S) \cdot P(M \mid S, L).
$$

Pentru o observație $(O=o, L=l, M=m)$, probabilitatea a posteriori ca e-mail-ul să fie spam este:
$$
P(S=1 \mid O=o, L=l, M=m) = \frac{P(S=1, O=o, L=l, M=m)}{P(S=1, O=o, L=l, M=m) + P(S=0, O=o, L=l, M=m)}.
$$

Folosind **regula de decizie MAP (Maximum A Posteriori)** cu prag $0.5$, clasificăm e-mail-ul ca **Spam ($S=1$)** dacă $P(S=1 \mid O, L, M) > 0.5$ (echivalent $P(S=1, O, L, M) > P(S=0, O, L, M)$), și **Non-Spam ($S=0$)** în caz contrar.

Evaluând toate cele $2^3 = 8$ configurații posibile ale atributelor $(O, L, M)$:

| $O$ (Ofertă) | $L$ (Link) | $M$ (Lung) | $P(S=1, O, L, M)$ | $P(S=0, O, L, M)$ | $P(S=1 \mid O, L, M)$ | Clasificare |
| :---: | :---: | :---: | :---: | :---: | :---: | :---: |
| $0$ | $0$ | $0$ | $0.4 \cdot 0.3 \cdot 0.2 \cdot 0.5 = 0.0120$ | $0.6 \cdot 0.9 \cdot 0.7 \cdot 0.8 = 0.3024$ | $\frac{5}{131} \approx 0.0382$ | **Non-Spam ($S=0$)** |
| $0$ | $0$ | $1$ | $0.4 \cdot 0.3 \cdot 0.2 \cdot 0.5 = 0.0120$ | $0.6 \cdot 0.9 \cdot 0.7 \cdot 0.2 = 0.0756$ | $\frac{10}{73} \approx 0.1370$ | **Non-Spam ($S=0$)** |
| $0$ | $1$ | $0$ | $0.4 \cdot 0.3 \cdot 0.8 \cdot 0.1 = 0.0096$ | $0.6 \cdot 0.9 \cdot 0.3 \cdot 0.4 = 0.0648$ | $\frac{4}{31} \approx 0.1290$ | **Non-Spam ($S=0$)** |
| $0$ | $1$ | $1$ | $0.4 \cdot 0.3 \cdot 0.8 \cdot 0.9 = 0.0864$ | $0.6 \cdot 0.9 \cdot 0.3 \cdot 0.6 = 0.0972$ | $\frac{8}{17} \approx 0.4706$ | **Non-Spam ($S=0$)** |
| $1$ | $0$ | $0$ | $0.4 \cdot 0.7 \cdot 0.2 \cdot 0.5 = 0.0280$ | $0.6 \cdot 0.1 \cdot 0.7 \cdot 0.8 = 0.0336$ | $\frac{5}{11} \approx 0.4545$ | **Non-Spam ($S=0$)** |
| $1$ | $0$ | $1$ | $0.4 \cdot 0.7 \cdot 0.2 \cdot 0.5 = 0.0280$ | $0.6 \cdot 0.1 \cdot 0.7 \cdot 0.2 = 0.0084$ | $\frac{10}{13} \approx 0.7692$ | **Spam ($S=1$)** |
| $1$ | $1$ | $0$ | $0.4 \cdot 0.7 \cdot 0.8 \cdot 0.1 = 0.0224$ | $0.6 \cdot 0.1 \cdot 0.3 \cdot 0.4 = 0.0072$ | $\frac{28}{37} \approx 0.7568$ | **Spam ($S=1$)** |
| $1$ | $1$ | $1$ | $0.4 \cdot 0.7 \cdot 0.8 \cdot 0.9 = 0.2016$ | $0.6 \cdot 0.1 \cdot 0.3 \cdot 0.6 = 0.0108$ | $\frac{56}{59} \approx 0.9492$ | **Spam ($S=1$)** |

### Regula logică echivalentă de clasificare
Din tabelul de mai sus rezultă o regulă booleană foarte clară:
$$
\text{Clasificare}(O, L, M) = \text{Spam} \iff O = 1 \;\land\; (L = 1 \;\lor\; M = 1).
$$
Cu alte cuvinte, un e-mail este clasificat drept **Spam** dacă și numai dacă **conține cuvântul „ofertă” ($O=1$) ȘI are cel puțin una dintre celelalte două caracteristici (conține link-uri $L=1$ sau are lungime mare $M=1$)**.

In [ ]:
import itertools
import pandas as pd

# Implementare cu pgmpy (și verificare analitică directă)
try:
    try:
        from pgmpy.models import DiscreteBayesianNetwork as BayesianNetwork
    except ImportError:
        from pgmpy.models import BayesianNetwork
    from pgmpy.factors.discrete import TabularCPD
    from pgmpy.inference import VariableElimination

    model = BayesianNetwork([('S', 'O'), ('S', 'L'), ('S', 'M'), ('L', 'M')])

    cpd_S = TabularCPD(variable='S', variable_card=2, values=[[0.6], [0.4]])
    cpd_O = TabularCPD(
        variable='O', variable_card=2,
        values=[[0.9, 0.3],
                [0.1, 0.7]],
        evidence=['S'], evidence_card=[2]
    )
    cpd_L = TabularCPD(
        variable='L', variable_card=2,
        values=[[0.7, 0.2],
                [0.3, 0.8]],
        evidence=['S'], evidence_card=[2]
    )
    # Ordinea coloanelor pentru evidence=['S', 'L']: (0,0), (0,1), (1,0), (1,1)
    cpd_M = TabularCPD(
        variable='M', variable_card=2,
        values=[[0.8, 0.4, 0.5, 0.1],
                [0.2, 0.6, 0.5, 0.9]],
        evidence=['S', 'L'], evidence_card=[2, 2]
    )

    model.add_cpds(cpd_S, cpd_O, cpd_L, cpd_M)
    assert model.check_model(), "Modelul Bayesian nu este valid!"

    print("=== Independențele locale din rețea (pgmpy) ===")
    print(model.local_independencies(['S', 'O', 'L', 'M']))
    print("\n=== Toate independențele (d-separare) ===")
    print(model.get_independencies())

    infer = VariableElimination(model)
    rows = []
    for o, l, m in itertools.product([0, 1], repeat=3):
        q = infer.query(variables=['S'], evidence={'O': o, 'L': l, 'M': m})
        p_s1 = float(q.values[1])
        rows.append({
            'O': o, 'L': l, 'M': m,
            'P(S=0 | O,L,M)': round(1 - p_s1, 6),
            'P(S=1 | O,L,M)': round(p_s1, 6),
            'Clasificare': 'SPAM (S=1)' if p_s1 > 0.5 else 'NON-SPAM (S=0)'
        })

    df_rezultate = pd.DataFrame(rows)
    print("\n=== Tabelul complet de clasificare ===")
    print(df_rezultate.to_string(index=False))

except ImportError:
    print("Notă: Librăria pgmpy nu este instalată în acest mediu. Se rulează calculul exact prin factorizare:")
    P_S = {0: 0.6, 1: 0.4}
    P_O = {0: 0.1, 1: 0.7}
    P_L = {0: 0.3, 1: 0.8}
    P_M = {(0, 0): 0.2, (0, 1): 0.6, (1, 0): 0.5, (1, 1): 0.9}

    rows = []
    for o, l, m in itertools.product([0, 1], repeat=3):
        p1 = P_S[1] * (P_O[1] if o else 1 - P_O[1]) * (P_L[1] if l else 1 - P_L[1]) * (P_M[(1, l)] if m else 1 - P_M[(1, l)])
        p0 = P_S[0] * (P_O[0] if o else 1 - P_O[0]) * (P_L[0] if l else 1 - P_L[0]) * (P_M[(0, l)] if m else 1 - P_M[(0, l)])
        post1 = p1 / (p1 + p0)
        rows.append({
            'O': o, 'L': l, 'M': m,
            'P(S=0 | O,L,M)': round(1 - post1, 6),
            'P(S=1 | O,L,M)': round(post1, 6),
            'Clasificare': 'SPAM (S=1)' if post1 > 0.5 else 'NON-SPAM (S=0)'
        })
    print(pd.DataFrame(rows).to_string(index=False))